# Sovereign CCA Dataset Builder

Combines QEDS debt data + MSCI indices for structural credit risk model.

**Frequency:** Weekly  
**Debt barrier:** Forward-filled from most recent quarter

**Inputs:**
- `data/processed/WB_QEDS/QEDS_SDDS.csv`
- `data/processed/MSCI_indices/mscicountryindex.csv`

**Output:**
- Weekly panel with: country, date, msci_index, msci_vol, debt_st, debt_lt, default_barrier

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Paths
QEDS_PATH = "data/processed/WB_QEDS/QEDS_SDDS.csv"
MSCI_PATH = "data/processed/MSCI_indices/mscicountryindex.csv"
OUTPUT_PATH = "data/processed/cca_panel_weekly.csv"

---
## 1. Load QEDS (Quarterly Debt Data)

In [4]:
qeds_raw = pd.read_csv(QEDS_PATH, delimiter=';')
print(f"Shape: {qeds_raw.shape}")
print(f"Columns: {qeds_raw.columns[:5].tolist()} ... {qeds_raw.columns[-3:].tolist()}")
qeds_raw.head(3)

Shape: (232200, 121)
Columns: ['Country Name', 'Country Code', 'Cleaned_Name', 'Indicator Name', 'Metric'] ... ['2024Q4', '2025Q1', '2025Q2']


,Country Name,Country Code,Cleaned_Name,Indicator Name,Metric,Sector,Maturity,Instrument,Currency,Filter1,...,2023Q1,2023Q2,2023Q3,2023Q4,2024Q1,2024Q2,2024Q3,2024Q4,2025Q1,2025Q2
0,Afghanistan,AFG,Afghanistan,"Ext. Assets in Debt Instruments, All Sectors, ...",Ext. Assets in Debt Instruments,All Sectors,All maturities,All instruments,USD,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,Afghanistan,"Ext. Assets in Debt Instruments, Central Bank,...",Ext. Assets in Debt Instruments,Central Bank,All maturities,All instruments,USD,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,Afghanistan,"Ext. Assets in Debt Instruments, Central Bank,...",Ext. Assets in Debt Instruments,Central Bank,Long-term,All instruments,USD,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# Check indicator names - find General Government debt
indicators = qeds_raw['Indicator Name'].unique()
gg_indicators = [i for i in indicators if 'General Government' in str(i)]
print(f"Found {len(gg_indicators)} General Government indicators")
for i in gg_indicators[:15]:
    print(f"  {i}")

Found 366 General Government indicators
  Ext. Assets in Debt Instruments, General Government, All maturities, All instruments, USD
  Ext. Assets in Debt Instruments, General Government, Long-term, All instruments, USD
  Ext. Assets in Debt Instruments, General Government, Long-term, Currency and deposits, USD
  Ext. Assets in Debt Instruments, General Government, Long-term, Debt securities, USD
  Ext. Assets in Debt Instruments, General Government, Long-term, Loans, USD
  Ext. Assets in Debt Instruments, General Government, Long-term, Other debt instruments, USD
  Ext. Assets in Debt Instruments, General Government, Long-term, Special drawing rights (SDRs), USD
  Ext. Assets in Debt Instruments, General Government, Long-term, Trade credit and advances, USD
  Ext. Assets in Debt Instruments, General Government, Short-term, All instruments, USD
  Ext. Assets in Debt Instruments, General Government, Short-term, Currency and deposits, USD
  Ext. Assets in Debt Instruments, General Governm

In [6]:
# Define which indicators to extract
# ADJUST THESE TO MATCH EXACT STRINGS IN YOUR DATA
DEBT_ST_INDICATOR = 'Gross Ext. Debt Pos., General Government, Short-term, All instruments, USD'
DEBT_LT_INDICATOR = 'Gross Ext. Debt Pos., General Government, Long-term, All instruments, USD'

# Check if they exist
print(f"ST exists: {DEBT_ST_INDICATOR in indicators}")
print(f"LT exists: {DEBT_LT_INDICATOR in indicators}")

ST exists: True
LT exists: True


In [7]:
# Identify quarter columns
id_cols = ['Cleaned_Name', 'Indicator Name']
quarter_cols = [c for c in qeds_raw.columns if c not in id_cols]
print(f"Quarter columns: {quarter_cols[:3]} ... {quarter_cols[-3:]}")
print(f"Total quarters: {len(quarter_cols)}")

Quarter columns: ['Country Name', 'Country Code', 'Metric'] ... ['2024Q4', '2025Q1', '2025Q2']
Total quarters: 119


In [8]:
# Filter to debt indicators and reshape
debt_indicators = [DEBT_ST_INDICATOR, DEBT_LT_INDICATOR]
qeds_debt = qeds_raw[qeds_raw['Indicator Name'].isin(debt_indicators)].copy()
print(f"Filtered to {len(qeds_debt)} rows")

# Melt to long format
qeds_long = qeds_debt.melt(
    id_vars=id_cols,
    value_vars=quarter_cols,
    var_name='quarter',
    value_name='value'
)
qeds_long.columns = ['country', 'indicator', 'quarter', 'value']
qeds_long['value'] = pd.to_numeric(qeds_long['value'], errors='coerce')
print(f"Long format: {len(qeds_long)} rows")
qeds_long.head()

Filtered to 258 rows
Long format: 30702 rows


,country,indicator,quarter,value
0,Afghanistan,"Gross Ext. Debt Pos., General Government, Long...",Country Name,NaN
1,Afghanistan,"Gross Ext. Debt Pos., General Government, Shor...",Country Name,NaN
2,Albania,"Gross Ext. Debt Pos., General Government, Long...",Country Name,NaN
3,Albania,"Gross Ext. Debt Pos., General Government, Shor...",Country Name,NaN
4,Algeria,"Gross Ext. Debt Pos., General Government, Long...",Country Name,NaN


In [ ]:
# Parse quarter to date (end of quarter)
def quarter_to_date(q):
    """Convert '2020Q1' to datetime 2020-03-31"""
    try:
        year = int(q[:4])
        qtr = int(q[-1])
        month = qtr * 3
        return pd.Timestamp(year=year, month=month, day=1) + pd.offsets.MonthEnd(0)
    except:
        return pd.NaT

qeds_long['date'] = qeds_long['quarter'].apply(quarter_to_date)
qeds_long = qeds_long.dropna(subset=['date'])
print(f"Date range: {qeds_long['date'].min()} to {qeds_long['date'].max()}")

In [ ]:
# Pivot: one column per indicator
qeds_wide = qeds_long.pivot_table(
    index=['country', 'date'],
    columns='indicator',
    values='value',
    aggfunc='first'
).reset_index()

# Rename columns
qeds_wide.columns.name = None
col_map = {
    DEBT_ST_INDICATOR: 'debt_st',
    DEBT_LT_INDICATOR: 'debt_lt'
}
qeds_wide = qeds_wide.rename(columns=col_map)

# Compute default barrier: ST + 0.5 * LT
qeds_wide['default_barrier'] = qeds_wide['debt_st'].fillna(0) + 0.5 * qeds_wide['debt_lt'].fillna(0)

print(f"Quarterly debt panel: {len(qeds_wide)} obs, {qeds_wide['country'].nunique()} countries")
qeds_wide.head()

---
## 2. Load MSCI (Daily Equity Indices)

In [ ]:
msci_raw = pd.read_csv(MSCI_PATH)
print(f"Shape: {msci_raw.shape}")
print(f"Columns: {msci_raw.columns[:5].tolist()}")
msci_raw.head(3)

In [ ]:
# Parse date column (first column, format DD.MM.YYYY)
date_col = msci_raw.columns[0]
msci_raw['date'] = pd.to_datetime(msci_raw[date_col], format='%d.%m.%Y', errors='coerce')

# If that fails, try dayfirst
if msci_raw['date'].isna().sum() > len(msci_raw) * 0.5:
    msci_raw['date'] = pd.to_datetime(msci_raw[date_col], dayfirst=True, errors='coerce')

msci_raw = msci_raw.drop(columns=[date_col])
print(f"Date range: {msci_raw['date'].min()} to {msci_raw['date'].max()}")

In [ ]:
# Melt to long format
country_cols = [c for c in msci_raw.columns if c != 'date']
print(f"Countries in MSCI: {len(country_cols)}")
print(f"Sample: {country_cols[:10]}")

msci_long = msci_raw.melt(
    id_vars=['date'],
    value_vars=country_cols,
    var_name='country',
    value_name='msci_index'
)
msci_long['msci_index'] = pd.to_numeric(msci_long['msci_index'], errors='coerce')
msci_long = msci_long.dropna(subset=['date', 'msci_index'])
print(f"Daily MSCI: {len(msci_long)} observations")

---
## 3. Resample MSCI to Weekly

In [ ]:
# Sort and compute daily log returns
msci_long = msci_long.sort_values(['country', 'date'])
msci_long['log_ret'] = msci_long.groupby('country')['msci_index'].transform(
    lambda x: np.log(x / x.shift(1))
)

# Assign week-ending Friday
msci_long['week'] = msci_long['date'] - pd.to_timedelta(msci_long['date'].dt.dayofweek - 4, unit='D')
# Adjust: if day is Sat/Sun, push to next Friday
msci_long['week'] = msci_long['date'].apply(
    lambda d: d + pd.Timedelta(days=(4 - d.dayofweek) % 7) if d.dayofweek <= 4 
              else d + pd.Timedelta(days=(4 - d.dayofweek + 7))
)

msci_long.head()

In [ ]:
# Aggregate to weekly
msci_weekly = msci_long.groupby(['country', 'week']).agg(
    msci_index=('msci_index', 'last'),           # End-of-week level
    msci_ret_weekly=('log_ret', 'sum'),          # Weekly return (sum of daily log returns)
    msci_vol_daily=('log_ret', 'std'),           # Daily vol within week
    n_days=('log_ret', 'count')
).reset_index()

msci_weekly.columns = ['country', 'date', 'msci_index', 'msci_ret_weekly', 'msci_vol_daily', 'n_days']

# Annualized volatility (rolling 52-week)
msci_weekly = msci_weekly.sort_values(['country', 'date'])
msci_weekly['msci_vol_annual'] = msci_weekly.groupby('country')['msci_ret_weekly'].transform(
    lambda x: x.rolling(window=52, min_periods=12).std() * np.sqrt(52)
)

print(f"Weekly MSCI: {len(msci_weekly)} observations, {msci_weekly['country'].nunique()} countries")
msci_weekly.head()

---
## 4. Expand Quarterly Debt to Weekly (Forward Fill)

In [ ]:
# Get all weeks from MSCI data
all_weeks = msci_weekly[['date']].drop_duplicates().sort_values('date')
all_countries_qeds = qeds_wide['country'].unique()

print(f"Weeks in MSCI: {len(all_weeks)}")
print(f"Countries in QEDS: {len(all_countries_qeds)}")

In [ ]:
# For each country, create weekly dates and forward-fill quarterly debt
debt_weekly_list = []

for country in all_countries_qeds:
    # Get quarterly debt for this country
    country_debt = qeds_wide[qeds_wide['country'] == country].copy()
    
    if len(country_debt) == 0:
        continue
    
    # Create weekly index spanning debt data range
    min_date = country_debt['date'].min()
    max_date = country_debt['date'].max()
    
    # Get weeks within this range from MSCI
    weeks = all_weeks[(all_weeks['date'] >= min_date) & 
                      (all_weeks['date'] <= max_date + pd.Timedelta(days=100))]['date'].tolist()
    
    if len(weeks) == 0:
        continue
    
    # Create weekly dataframe
    weekly_df = pd.DataFrame({'date': weeks, 'country': country})
    
    # Merge with quarterly debt (will have NaN for non-quarter-end weeks)
    weekly_df = weekly_df.merge(
        country_debt[['date', 'debt_st', 'debt_lt', 'default_barrier']],
        on='date',
        how='left'
    )
    
    # Forward fill
    weekly_df = weekly_df.sort_values('date')
    weekly_df[['debt_st', 'debt_lt', 'default_barrier']] = weekly_df[['debt_st', 'debt_lt', 'default_barrier']].ffill()
    
    debt_weekly_list.append(weekly_df)

debt_weekly = pd.concat(debt_weekly_list, ignore_index=True)
print(f"Weekly debt panel: {len(debt_weekly)} obs, {debt_weekly['country'].nunique()} countries")
debt_weekly.head(10)

---
## 5. Merge MSCI + Debt

In [ ]:
# Standardize country names for matching
debt_weekly['country_clean'] = debt_weekly['country'].str.strip().str.lower()
msci_weekly['country_clean'] = msci_weekly['country'].str.strip().str.lower()

# Check overlap
debt_countries = set(debt_weekly['country_clean'].unique())
msci_countries = set(msci_weekly['country_clean'].unique())
overlap = debt_countries & msci_countries

print(f"QEDS countries: {len(debt_countries)}")
print(f"MSCI countries: {len(msci_countries)}")
print(f"Overlap: {len(overlap)}")

if len(overlap) < 5:
    print("\nWARNING: Low overlap! Sample names:")
    print(f"  QEDS: {list(debt_countries)[:10]}")
    print(f"  MSCI: {list(msci_countries)[:10]}")

In [ ]:
# Merge
panel = msci_weekly.merge(
    debt_weekly,
    on=['country_clean', 'date'],
    how='inner'
)

# Clean up columns
panel['country'] = panel['country_x']
panel = panel.drop(columns=['country_x', 'country_y', 'country_clean'])

# Reorder
cols = ['country', 'date', 'msci_index', 'msci_ret_weekly', 'msci_vol_annual', 
        'debt_st', 'debt_lt', 'default_barrier', 'n_days']
panel = panel[[c for c in cols if c in panel.columns]]

print(f"Final panel: {len(panel)} obs, {panel['country'].nunique()} countries")
panel.head()

---
## 6. Summary & Export

In [ ]:
print("="*60)
print("PANEL SUMMARY")
print("="*60)
print(f"Date range: {panel['date'].min()} to {panel['date'].max()}")
print(f"Countries: {panel['country'].nunique()}")
print(f"Total observations: {len(panel):,}")
print(f"\nCountries: {sorted(panel['country'].unique())}")
print(f"\nMissing values:")
print(panel.isnull().sum())
print(f"\nDescriptive stats:")
panel.describe()

In [ ]:
# Save
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

In [ ]:
# Quick plot for sanity check
import matplotlib.pyplot as plt

sample_country = panel['country'].value_counts().index[0]
sample = panel[panel['country'] == sample_country].copy()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(sample['date'], sample['msci_index'])
axes[0].set_ylabel('MSCI Index')
axes[0].set_title(f'{sample_country}: Asset Proxy & Default Barrier')

axes[1].plot(sample['date'], sample['default_barrier'] / 1e9, label='Default Barrier')
axes[1].set_ylabel('Barrier (USD bn)')
axes[1].set_xlabel('Date')
axes[1].legend()

plt.tight_layout()
plt.show()